In [1]:
#load modules

import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import os
import networkx as nx
import json
import ndjson
from urllib.request import urlopen
from networkx import betweenness_centrality
from networkx import density

ModuleNotFoundError: No module named 'ndjson'

In [2]:
!pip install ndjson

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
home_dir = '/content/gdrive/MyDrive/KB RiR/Publicaties/DutchDraCor_gender/'
output_folder = home_dir + 'data'
os.listdir(home_dir)

['Abstract Lassche and Van der Deijl DH Benelux 2025.docx',
 'Notebooks',
 'Artikelen',
 'data',
 'Klad_gender.docx',
 'Figuren',
 'tables_gender_speech_EMLC.xlsx',
 'Concept artikel.gdoc']

# Part 1: Data extraction

In [ ]:
# Read metadatafile

corpus = "dutch" # 'fre', 'dutch', 'eng', 'ger'

URL_metadata = "https://dracor.org/api/v1/corpora/" + corpus + "/metadata/csv"

with urlopen(URL_metadata) as metadata_file:
  metadata_table = pd.read_csv(metadata_file)

URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

In [ ]:
# filter

list_of_titles = []

for index, row in metadata_table.iterrows():
  name = row['name']
  year = row['yearNormalized']
  if year >= 1500 and year < 1800:
    list_of_titles.append(name)

print(len(list_of_titles))

1343


In [ ]:
def create_speakersdict(root_play):
    speakers_dict = {}
    speaker_list = [persona for persona in root_play.findall(".//{http://www.tei-c.org/ns/1.0}personGrp") + root_play.findall(".//{http://www.tei-c.org/ns/1.0}person")]
    for speaker in speaker_list:
        speaker_id = speaker.get("{http://www.w3.org/XML/1998/namespace}id")
        if speaker.find(".//{http://www.tei-c.org/ns/1.0}name") != None:
          name = speaker.find(".//{http://www.tei-c.org/ns/1.0}name").text
        else:
          name = speaker.find(".//{http://www.tei-c.org/ns/1.0}persName").text
        speakers_dict[speaker_id] = {}
        speakers_dict[speaker_id]["name"] = name
        speakers_dict[speaker_id]["gender"] = speaker.get("sex")
        speakers_dict[speaker_id]["all_lines"] = []
    return(speakers_dict)

In [ ]:
all_speech_turns = 0
male_speech_turns = 0
female_speech_turns = 0

male_lines = []
female_lines = []

speakers_database = {}

for title in list_of_titles:
  URL_play = "https://dracor.org/api/corpora/"+ corpus + "/play/" + title + "/tei"
  with urlopen(URL_play) as TEI_file:
    tree = ET.parse(TEI_file)
    root_play = tree.getroot()

  #tei_header = root_play.find(".//{http://www.tei-c.org/ns/1.0}teiHeader")
  body = root_play.find(".//{http://www.tei-c.org/ns/1.0}body")
  speakers = create_speakersdict(root_play)

  #print(speakers)

  for sp in root_play.findall(".//{http://www.tei-c.org/ns/1.0}sp"):
    all_speech_turns += 1
    if sp.get("who") != None:
      speaker_id = sp.get("who").replace("#", "")
    if speaker_id in speakers:
      gender = speakers[speaker_id]["gender"]
      if speaker_id not in speakers_database:
        speakers_database[speaker_id] = {}
        speakers_database[speaker_id]["speaker"] = speakers[speaker_id]["name"]
        speakers_database[speaker_id]["gender"] = gender
        speakers_database[speaker_id]["speech"] = ''
        speakers_database[speaker_id]["play"] = title
      lines = sp.findall(".//{http://www.tei-c.org/ns/1.0}lb") + sp.findall(".//{http://www.tei-c.org/ns/1.0}l") + sp.findall(".//{http://www.tei-c.org/ns/1.0}s") + sp.findall(".//{http://www.tei-c.org/ns/1.0}p")

      lines_in_sp = [line.text + " " for line in lines if line.text != None]

      if len(lines) > 0:
        if gender == "MALE":
          male_speech_turns += 1

          for line in lines:
            if line.text != None:
              male_lines.append(line.text)

        if gender == "FEMALE":
          female_speech_turns += 1
          for line in lines:
            if line.text != None:
              female_lines.append(line.text)

        speakers_database[speaker_id]["speech"] = speakers_database[speaker_id]["speech"] + " ".join(lines_in_sp).replace("  ", " ")
  print("Speech turns extracted from", title)

print(all_speech_turns)
print(male_speech_turns)
print(female_speech_turns)

Speech turns extracted from abeille-argelie
Speech turns extracted from abeille-coriolan
Speech turns extracted from abeille-lyncee
Speech turns extracted from aigueberre-avare-amoureux
Speech turns extracted from aigueberre-pan-et-doris
Speech turns extracted from aigueberre-polixene
Speech turns extracted from aigueberre-prologue
Speech turns extracted from alain-legrand-epreuve-reciproque
Speech turns extracted from allainval-ecole-des-bourgeois
Speech turns extracted from allainval-hiver
Speech turns extracted from andrieux-anaximandre
Speech turns extracted from andrieux-les-etourdis
Speech turns extracted from anomyme-dialogue-de-la-prude-et-de-la-coquette
Speech turns extracted from anonyme-bonnes-gens
Speech turns extracted from anonyme-chapelain-decoiffe-2
Speech turns extracted from anonyme-chapelain-decoiffe
Speech turns extracted from anonyme-club-des-dames
Speech turns extracted from anonyme-d-alcippe-et-de-drionice
Speech turns extracted from anonyme-deroute-des-precieuse

In [ ]:
speakers_database

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
speakers_rows = []

for speaker in speakers_database:
  speakers_rows.append(speakers_database[speaker])

database_as_string = str(speakers_rows) #.replace('\'speaker\'', '\"speaker\"').replace('\'play\'', '\"play\"').replace('\'gender\'', '\"gender\"').replace('\'speech\'', '\"speech\"').replace('None', "")
#input = database_as_string
input = json.dumps(speakers_rows, ensure_ascii=False).encode('utf8')
data = json.loads(input)
output = ndjson.dumps(data, ensure_ascii=False).encode('utf8')
#print(output)

In [ ]:
 output_filename = "speech_gender_" + corpus + ".ndjson"

with open(output_folder + "/" + output_filename, 'w', encoding='utf8') as f:
    ndjson.dump(data, f, ensure_ascii=False)

# Part 2 Analysis

In [ ]:
corpora = ["fre", "ger", "eng", "dutch", "ita", "cal"]
plays_database = {}

for corpus_code in corpora:

  dataset = pd.read_json(output_folder + "/speech_gender_" + corpus_code + ".ndjson", lines=True)

  plays_database[corpus_code] = {}

  for index, row in dataset.iterrows():
    character_id = row["speaker"]
    character_gender = row["gender"]
    play_id = row["play"]
    character_speech = row["speech"]
    character_tokens = len(character_speech.split())
    if play_id not in plays_database[corpus_code]:
      plays_database[corpus_code][play_id] = {}
      plays_database[corpus_code][play_id]["speakers"] = 0
      plays_database[corpus_code][play_id]["tokens"] = 0
      plays_database[corpus_code][play_id]["male_chars"] = 0
      plays_database[corpus_code][play_id]["male_tokens"] = 0
      plays_database[corpus_code][play_id]["female_chars"] = 0
      plays_database[corpus_code][play_id]["female_tokens"] = 0
      plays_database[corpus_code][play_id]["unknown_chars"] = 0
      plays_database[corpus_code][play_id]["unknown_tokens"] = 0

    plays_database[corpus_code][play_id]["tokens"] += character_tokens
    plays_database[corpus_code][play_id]["speakers"] += 1

    if character_gender == "MALE":
      plays_database[corpus_code][play_id]["male_chars"] += 1
      plays_database[corpus_code][play_id]["male_tokens"] += character_tokens
    if character_gender == "FEMALE":
      plays_database[corpus_code][play_id]["female_chars"] += 1
      plays_database[corpus_code][play_id]["female_tokens"] += character_tokens
    if character_gender == "UNKNOWN":
      plays_database[corpus_code][play_id]["unknown_chars"] += 1
      plays_database[corpus_code][play_id]["unknown_tokens"] += character_tokens

  male_speakers_freqs = []
  female_speakers_freqs = []
  unknown_speakers_freqs = []

  male_tokens_freqs = []
  female_tokens_freqs = []
  unknown_tokens_freqs = []

  for play_id in plays_database[corpus_code]:
    all_speakers = plays_database[corpus_code][play_id]["speakers"]
    all_tokens = plays_database[corpus_code][play_id]["tokens"]
    male_speakers_rel = plays_database[corpus_code][play_id]["male_chars"] / all_speakers
    female_speakers_rel = plays_database[corpus_code][play_id]["female_chars"] / all_speakers
    unknown_speakers_rel = plays_database[corpus_code][play_id]["unknown_chars"] / all_speakers

    male_tokens_rel = plays_database[corpus_code][play_id]["male_tokens"] / all_tokens
    female_tokens_rel = plays_database[corpus_code][play_id]["female_tokens"] / all_tokens
    unknown_tokens_rel = plays_database[corpus_code][play_id]["unknown_tokens"] / all_tokens

    male_speakers_freqs.append(male_speakers_rel)
    female_speakers_freqs.append(female_speakers_rel)
    unknown_speakers_freqs.append(unknown_speakers_rel)

    male_tokens_freqs.append(male_tokens_rel)
    female_tokens_freqs.append(female_tokens_rel)
    unknown_tokens_freqs.append(unknown_tokens_rel)

    plays_database[corpus_code][play_id]["male_rel_chars"] = male_speakers_freqs
    plays_database[corpus_code][play_id]["female_rel_chars"] = female_speakers_freqs
    plays_database[corpus_code][play_id]["unknown_rel_chars"] = unknown_speakers_freqs

    plays_database[corpus_code][play_id]["male_rel_tokens"] = male_tokens_freqs
    plays_database[corpus_code][play_id]["female_rel_tokens"] = female_tokens_freqs
    plays_database[corpus_code][play_id]["unknown_rel_tokens"] = unknown_tokens_freqs

In [ ]:
plays_database

Buffered data was truncated after reaching the output size limit.

In [ ]:
female_tokens_freqs

[0.598794073476341,
 0.26166872086169013,
 0.20704438524281418,
 0.39702282764859254,
 0.30951301493003985,
 0.5308625336927224,
 0.6194354242908441,
 0.28278711937996304,
 0.363105231347794,
 0.15508424847011484,
 0.6724671551288529,
 0.0400764059802373,
 0.45442916533418615,
 0.16048355306381673,
 0.10326313594662219,
 0.34286591051258747,
 0.39987778633863813,
 0.49839644751449363,
 0.6220120830049908,
 0.08930521995299726,
 0.10950704225352113,
 0.4367878249449573,
 0.2376482213438735,
 0.24457810012382805,
 0.0014337229004648738,
 0.19163673678809645,
 0.05546132428713157,
 0.0,
 0.2833814163627456,
 0.31472311628290833,
 0.3930400419371749,
 0.23639403405341194,
 0.3520420640743458,
 0.10315101070154578,
 0.21752895752895754,
 0.5703191590112701,
 0.8390699656106394,
 0.0,
 0.35025049008930514,
 0.75723187930469,
 0.5101451421800948,
 0.6492938286766964,
 0.14696485623003194,
 0.855072463768116,
 0.24698228562470606,
 0.3078728886344117,
 0.9224155303284745,
 0.2697175141242938,
